# OC Transpo route-time bus allocation (Gurobi proof of concept)

This notebook mirrors the earlier SciPy proof of concept, but solves the route-time integer allocation model with **Gurobi**.

It:
- loads `Mar 23 Subset OC Transpo.csv`
- builds a simple integer model for buses per route and time block
- applies route-level lower/upper bounds, time-block fleet caps, and a budget cap
- solves with Gurobi and prints a few diagnostics that are useful for analysis


In [36]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

candidate_paths = [
    Path('Mar 23 Subset OC Transpo.csv'),
    Path('data/Mar 23 Subset OC Transpo.csv'),
    Path('/mnt/data/Mar 23 Subset OC Transpo.csv'),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find Mar 23 Subset OC Transpo.csv')

print(f'Using data file: {csv_path}')
df = pd.read_csv(csv_path)
df.head()


Using data file: data\Mar 23 Subset OC Transpo.csv


,Route,Time Block,Ridership,Route Length (km),Round Trip Time (hr),Cost per Bus ($/hr),Min Buses,Max Buses,Benefit per Bus
0,5,Early AM,23,12,1.16,120,1,6,1.15
1,5,AM Peak,292,12,1.16,120,2,12,14.60
2,5,Midday,566,12,1.16,120,2,14,28.30
3,5,PM Peak,303,12,1.16,120,2,12,15.20
4,5,Evening,162,12,1.16,120,1,8,8.10


In [37]:
# Assumptions for the proof-of-concept model

time_block_hours = {
    'Early AM': 4,
    'AM Peak': 3,
    'Midday': 6,
    'PM Peak': 3,
    'Evening': 5,
    'Night': 3,
}

fleet_cap = {
    'Early AM': 12,
    'AM Peak': 22,
    'Midday': 24,
    'PM Peak': 22,
    'Evening': 16,
    'Night': 10,
}

# Daily budget for this subset only
budget = 50000

# Preprocess
model_df = df.copy()
model_df['Block Hours'] = model_df['Time Block'].map(time_block_hours)

# Proxy objective coefficient:
# reward high-ridership route-time combinations, while discounting long round trips
model_df['Effective Benefit'] = model_df['Benefit per Bus'] / model_df['Round Trip Time (hr)']

# Daily cost of assigning one bus to that route-time combination
model_df['Unit Cost'] = model_df['Cost per Bus ($/hr)'] * model_df['Block Hours']

model_df[['Route','Time Block','Ridership','Min Buses','Max Buses',
          'Block Hours','Unit Cost','Effective Benefit']].head(10)


,Route,Time Block,Ridership,Min Buses,Max Buses,Block Hours,Unit Cost,Effective Benefit
0,5,Early AM,23,1,6,4,480,0.99
1,5,AM Peak,292,2,12,3,360,12.59
2,5,Midday,566,2,14,6,720,24.40
3,5,PM Peak,303,2,12,3,360,13.10
4,5,Evening,162,1,8,5,600,6.98
5,5,Night,73,1,6,3,360,3.15
6,6,Early AM,177,2,10,4,480,5.40
7,6,AM Peak,1724,4,20,3,360,52.56
8,6,Midday,3513,4,24,6,720,107.13
9,6,PM Peak,2781,4,22,3,360,84.82


In [38]:
# Feasibility diagnostics before solve

print('=== Feasibility diagnostics ===\n')

min_by_block = model_df.groupby('Time Block')['Min Buses'].sum()
max_by_block = model_df.groupby('Time Block')['Max Buses'].sum()

print('Min buses required by time block:')
print(min_by_block)
print('\nMax buses allowed by time block:')
print(max_by_block)

print(f'\nBudget = {budget:,.2f}')
min_cost = (model_df['Min Buses'] * model_df['Unit Cost']).sum()
print(f'Cost of minimum-service solution = {min_cost:,.2f}')
if min_cost > budget:
    print('WARNING: infeasible because minimum-service cost exceeds budget')

print('\nFleet caps by time block:')
for tb, cap in fleet_cap.items():
    req = min_by_block.get(tb, 0)
    if req > cap:
        print(f'WARNING: {tb}: min required {req} > cap {cap}')
    else:
        print(f'{tb}: min required {req}, cap {cap}, slack {cap - req}')


=== Feasibility diagnostics ===

Min buses required by time block:
Time Block
AM Peak     15
Early AM     7
Evening      8
Midday      15
Night        5
PM Peak     15
Name: Min Buses, dtype: int64

Max buses allowed by time block:
Time Block
AM Peak     76
Early AM    36
Evening     52
Midday      90
Night       40
PM Peak     82
Name: Max Buses, dtype: int64

Budget = 50,000.00
Cost of minimum-service solution = 31,560.00

Fleet caps by time block:
Early AM: min required 7, cap 12, slack 5
AM Peak: min required 15, cap 22, slack 7
Midday: min required 15, cap 24, slack 9
PM Peak: min required 15, cap 22, slack 7
Evening: min required 8, cap 16, slack 8
Night: min required 5, cap 10, slack 5


In [39]:
import gurobipy as gp
from gurobipy import GRB

# print('Gurobi version:', gp.gurobi.version())

start = time.time()

# import os

# # Remove WLS credentials if present
# os.environ.pop("WLSACCESSID", None)
# os.environ.pop("WLSSECRET", None)
# os.environ.pop("LICENSEID", None)

# # Force local license
# os.environ["GRB_LICENSE_FILE"] = r"C:\Users\joemc\gurobi.lic"

m = gp.Model('oc_transpo_bus_allocation')

# Decision variables: integer buses assigned to each route-time combination
x = {}
for i in model_df.index:
    x[i] = m.addVar(
        vtype=GRB.INTEGER,
        lb=int(model_df.loc[i, 'Min Buses']),
        ub=int(model_df.loc[i, 'Max Buses']),
        name=f"x_{int(model_df.loc[i, 'Route'])}_{model_df.loc[i, 'Time Block'].replace(' ', '_')}"
    )

# Objective: maximize proxy benefit
m.setObjective(
    gp.quicksum(model_df.loc[i, 'Effective Benefit'] * x[i] for i in model_df.index),
    GRB.MAXIMIZE
)

# Fleet cap by time block
for tb, cap in fleet_cap.items():
    idx = model_df.index[model_df['Time Block'] == tb]
    m.addConstr(gp.quicksum(x[i] for i in idx) <= cap, name=f'fleet_{tb.replace(" ", "_")}')

# Budget constraint
m.addConstr(
    gp.quicksum(model_df.loc[i, 'Unit Cost'] * x[i] for i in model_df.index) <= budget,
    name='budget'
)

# Useful logging / limits for analysis
m.Params.OutputFlag = 1
m.Params.MIPGap = 0.0

m.optimize()

solve_time = time.time() - start

if m.Status != GRB.OPTIMAL:
    raise RuntimeError(f'Gurobi did not return an optimal solution. Status = {m.Status}')

print('\n=== Gurobi Diagnostics ===')
print('Status              :', m.Status)
print(f'Objective value     : {m.ObjVal:,.4f}')
print(f'Best bound          : {m.ObjBound:,.4f}')
print(f'MIP gap             : {m.MIPGap:.6f}')
print(f'Node count          : {m.NodeCount}')
print(f'Runtime (s)         : {m.Runtime:.4f}')
print(f'Wall time (s)       : {solve_time:.4f}')


Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1195G7 @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0

Academic license 2796207 - for non-commercial use only - registered to jo___@cmail.carleton.ca
Optimize a model with 7 rows, 30 columns and 60 nonzeros
Model fingerprint: 0x8bf3f0b7
Variable types: 0 continuous, 30 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+01, 5e+04]
Found heuristic solution: objective 3443.5527642
Presolve removed 7 rows and 30 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of

In [40]:
# Attach solution back to the dataframe

model_df['Optimal Buses'] = [int(round(x[i].X)) for i in model_df.index]
model_df['Benefit Contribution'] = model_df['Effective Benefit'] * model_df['Optimal Buses']
model_df['Cost Contribution'] = model_df['Unit Cost'] * model_df['Optimal Buses']
model_df['At Min?'] = model_df['Optimal Buses'] == model_df['Min Buses']
model_df['At Max?'] = model_df['Optimal Buses'] == model_df['Max Buses']
model_df['Slack Above Min'] = model_df['Optimal Buses'] - model_df['Min Buses']
model_df['Slack Below Max'] = model_df['Max Buses'] - model_df['Optimal Buses']

solution_df = model_df[[
    'Route','Time Block','Ridership','Min Buses','Max Buses','Optimal Buses',
    'Unit Cost','Effective Benefit','Benefit Contribution','Cost Contribution',
    'At Min?','At Max?','Slack Above Min','Slack Below Max'
]].copy()

solution_df.sort_values(['Time Block','Route'])


,Route,Time Block,Ridership,Min Buses,Max Buses,Optimal Buses,Unit Cost,Effective Benefit,Benefit Contribution,Cost Contribution,At Min?,At Max?,Slack Above Min,Slack Below Max
1,5,AM Peak,292,2,12,2,360,12.59,25.17,720,True,False,0,10
7,6,AM Peak,1724,4,20,4,360,52.56,210.24,1440,True,False,0,16
13,7,AM Peak,1649,4,20,11,360,55.71,612.80,3960,False,False,7,9
19,9,AM Peak,383,2,10,2,360,19.15,38.30,720,True,False,0,8
25,10,AM Peak,879,3,14,3,360,33.30,99.89,1080,True,False,0,11
0,5,Early AM,23,1,6,1,480,0.99,0.99,480,True,False,0,5
6,6,Early AM,177,2,10,2,480,5.40,10.79,960,True,False,0,8
12,7,Early AM,110,2,8,2,480,3.72,7.43,960,True,False,0,6
18,9,Early AM,44,1,6,1,480,2.20,2.20,480,True,False,0,5
24,10,Early AM,42,1,6,1,480,1.59,1.59,480,True,False,0,5


In [41]:
# Summary checks and quick analysis

summary = {
    'Objective (proxy benefit)': model_df['Benefit Contribution'].sum(),
    'Objective from Gurobi': m.ObjVal,
    'Total cost': model_df['Cost Contribution'].sum(),
    'Budget': budget,
    'Slack in budget': budget - model_df['Cost Contribution'].sum(),
    'Variables at min': int(model_df['At Min?'].sum()),
    'Variables at max': int(model_df['At Max?'].sum()),
}

for tb in fleet_cap:
    used = model_df.loc[model_df['Time Block'] == tb, 'Optimal Buses'].sum()
    summary[f'Fleet used - {tb}'] = int(used)
    summary[f'Fleet cap - {tb}'] = int(fleet_cap[tb])
    summary[f'Fleet slack - {tb}'] = int(fleet_cap[tb] - used)

pd.Series(summary)


Objective (proxy benefit)    5,442.72
Objective from Gurobi        5,442.72
Total cost                  49,680.00
Budget                      50,000.00
Slack in budget                320.00
Variables at min                25.00
Variables at max                 0.00
Fleet used - Early AM            7.00
Fleet cap - Early AM            12.00
Fleet slack - Early AM           5.00
Fleet used - AM Peak            22.00
Fleet cap - AM Peak             22.00
Fleet slack - AM Peak            0.00
Fleet used - Midday             24.00
Fleet cap - Midday              24.00
Fleet slack - Midday             0.00
Fleet used - PM Peak            22.00
Fleet cap - PM Peak             22.00
Fleet slack - PM Peak            0.00
Fleet used - Evening            16.00
Fleet cap - Evening             16.00
Fleet slack - Evening            0.00
Fleet used - Night              10.00
Fleet cap - Night               10.00
Fleet slack - Night              0.00
dtype: float64

In [42]:
# Compact route-time table for easy copy/paste into the report

pivot = solution_df.pivot(index='Route', columns='Time Block', values='Optimal Buses')
pivot = pivot[['Early AM','AM Peak','Midday','PM Peak','Evening','Night']]
pivot


Time Block,Early AM,AM Peak,Midday,PM Peak,Evening,Night
Route,,,,,,
5,1,2,2,2,1,1
6,2,4,4,4,2,1
7,2,11,13,11,10,6
9,1,2,2,2,1,1
10,1,3,3,3,2,1


## Notes for the next iteration

Useful Gurobi quantities for your project analysis:
- `m.ObjVal`: final objective
- `m.ObjBound`: best bound found by the solver
- `m.MIPGap`: optimality gap
- `m.NodeCount`: branch-and-bound nodes explored
- `m.Runtime`: solver runtime

Good next experiments:
1. tighten or relax `budget` and see which constraints become binding
2. compare this Gurobi baseline against the earlier SciPy/HiGHS version
3. add one or two valid inequalities / cuts and compare runtime, node count, and gap
